In [1]:
import json
import torch
import pickle
from torch.utils.data import Dataset, DataLoader

# Install/Upgrade transformers, accelerate, torch, and torchvision
# !pip install -U transformers accelerate torch torchvision

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_scheduler
)
from torch.optim import AdamW # Corrected import statement, using torch.optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

# ================================================
# CONFIGURATION
# ================================================

DATASET_PATH = "/content/drive/MyDrive/DataSet/LatestProperDatasets/clf2completeDataset.json"
OUTPUT_DIR   = "/content/drive/MyDrive/Models/EVA/clf2"
MODEL_NAME   = "distilbert-base-uncased"
MAX_LEN      = 128
BATCH_SIZE   = 32
EPOCHS       = 5
LR           = 2e-5
VAL_SPLIT    = 0.2
SEED         = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ================================================
# LOAD DATASET
# ================================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

texts  = [item["text"] for item in raw_data]
labels = [item["intent"] for item in raw_data]

print(f"Total examples : {len(texts)}")
print(f"Unique intents : {len(set(labels))}")
print(f"Intents        : {sorted(set(labels))}")

# ================================================
# ENCODE LABELS
# ================================================

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)

print(f"\nLabel mapping:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i} → {cls}")

# ================================================
# TRAIN/VAL SPLIT
# ================================================

X_train, X_val, y_train, y_val = train_test_split(
    texts, encoded_labels,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=encoded_labels
)

print(f"\nTrain size : {len(X_train)}")
print(f"Val size   : {len(X_val)}")

# ================================================
# DATASET CLASS
# ================================================

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"      : self.encodings["input_ids"][idx],
            "attention_mask" : self.encodings["attention_mask"][idx],
            "labels"         : self.labels[idx]
        }

# ================================================
# TOKENIZER AND DATALOADERS
# ================================================

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

train_dataset = IntentDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = IntentDataset(X_val,   y_val,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# ================================================
# MODEL
# ================================================

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
)
model.to(DEVICE)

# ================================================
# OPTIMIZER AND SCHEDULER
# ================================================

optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# ================================================
# TRAIN AND EVAL FUNCTIONS
# ================================================

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels

# ================================================
# TRAINING LOOP
# ================================================

best_val_acc = 0
best_epoch   = 0

print("\nStarting training...")
print("=" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler
    )
    val_loss, val_acc, val_preds, val_labels = eval_epoch(
        model, val_loader
    )

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.4f}")
    print(f"  Val Loss   : {val_loss:.4f}   | Val Acc   : {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch + 1
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  ✓ Best model saved (val_acc: {val_acc:.4f})")

    print("-" * 60)

print(f"\nBest Val Accuracy : {best_val_acc:.4f} at epoch {best_epoch}")

# ================================================
# FINAL EVALUATION
# ================================================

print("\nFinal Evaluation:")
print("=" * 60)
_, _, final_preds, final_labels = eval_epoch(model, val_loader)
print(classification_report(
    final_labels,
    final_preds,
    target_names=label_encoder.classes_
))

# ================================================
# SAVE LABEL ENCODER
# ================================================

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(f"{OUTPUT_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print(f"Label encoder saved → {OUTPUT_DIR}/label_encoder.pkl")
print(f"Model saved         → {OUTPUT_DIR}")

Device: cuda
Total examples : 24229
Unique intents : 8
Intents        : ['add', 'behavioral', 'delete', 'learning_create', 'learning_delete', 'learning_update', 'show', 'update']

Label mapping:
  0 → add
  1 → behavioral
  2 → delete
  3 → learning_create
  4 → learning_delete
  5 → learning_update
  6 → show
  7 → update

Train size : 19383
Val size   : 4846


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

... (8 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (val_acc: 0.9998)
------------------------------------------------------------
Epoch 4/5
  Train Loss : 0.0007 | Train Acc : 1.0000
  Val Loss   : 0.0012   | Val Acc   : 0.9996
------------------------------------------------------------
Epoch 5/5
  Train Loss : 0.0005 | Train Acc : 1.0000
  Val Loss   : 0.0005   | Val Acc   : 1.0000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (val_acc: 1.0000)
------------------------------------------------------------

Best Val Accuracy : 1.0000 at epoch 5

Final Evaluation:
                 precision    recall  f1-score   support

            add       1.00      1.00      1.00       500
     behavioral       1.00      1.00      1.00       400
         delete       1.00      1.00      1.00       500
learning_create       1.00      1.00      1.00       993
learning_delete       1.00      1.00      1.00       398
learning_update       1.00      1.00      1.00       996
           show       1.00      1.00      1.00       500
         update       1.00      1.00      1.00       559

       accuracy                           1.00      4846
      macro avg       1.00      1.00      1.00      4846
   weighted avg       1.00      1.00      1.00      4846

Label encoder saved → /content/drive/MyDrive/Models/EVA/clf2/label_encoder.pkl
Model saved         → /content/drive/MyDrive/Models/EVA/clf2


In [ ]:
import torch
import pickle
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification
)

# ================================================
# CONFIGURATION
# ================================================

MODEL_DIR  = "/content/drive/MyDrive/Models/EVA/intent_classifier"
MAX_LEN    = 128
THRESHOLD  = 0.7

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ================================================
# LOAD MODEL
# ================================================

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
model     = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

with open(f"{MODEL_DIR}/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

print(f"Model loaded from {MODEL_DIR}")
print(f"Intents: {list(label_encoder.classes_)}")

# ================================================
# INFERENCE FUNCTION
# ================================================

def predict_intent(text: str, threshold: float = THRESHOLD) -> dict:
    encoding = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    input_ids      = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    probs      = torch.softmax(outputs.logits, dim=1)
    confidence = probs.max().item()
    pred_idx   = probs.argmax().item()
    intent     = label_encoder.inverse_transform([pred_idx])[0]

    return {
        "text"            : text,
        "intent"          : intent if confidence >= threshold else "OUT_OF_SCOPE",
        "confidence"      : round(confidence, 4),
        "above_threshold" : confidence >= threshold
    }

Device: cuda


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded from /content/drive/MyDrive/Models/EVA/intent_classifier
Intents: [np.str_('add'), np.str_('behavioral'), np.str_('delete'), np.str_('learning_create'), np.str_('show'), np.str_('update')]


In [ ]:

# ================================================
# TEST
# ================================================

test_inputs = [
    "show my productivity report"
]

print("\nInference Results:")
print("=" * 60)
for text in test_inputs:
    result = predict_intent(text)
    status = "✓" if result["above_threshold"] else "⚠ LOW CONF"
    print(f"[{status}] {result['text']}")
    print(f"       Intent     : {result['intent']}")
    print(f"       Confidence : {result['confidence']}")
    print("-" * 40)



Inference Results:
[✓] show my productivity report
       Intent     : behavioral
       Confidence : 0.9993
----------------------------------------


In [ ]:
test_inputs = [
    # implicit task name
    "Physics exam is on Monday at 5PM",

    # casual language
    "yo remove that physics thing i added yesterday",

    # spelling mistakes
    "schedule a meeting with hamid on wednesday at 3pm",

    # multiple entities
    "move my Math exam on Friday to Monday at 6PM",

    # collaborator + date + time
    "add a group study session with Ahmed and Sara on Sunday at 4PM",

    # learning with go through pattern
    "I need to go through Python arrays starting next Monday in 2 weeks",

    # no trigger word — implicit create
    "MongoDB queries next Friday 1 month",

    # update roadmap — multiple fields
    "change my Python arrays scheduled date to next Monday and mark it ongoing",

    # show with vague language
    "what do I have going on this week",

    # delete with implicit reference
    "I am done with JavaScript I don't want it anymore",

    # collaborator with learning
    "learn Python basics with Hamid in 2 weeks starting Friday",

    # priority + status + date
    "show my high priority pending tasks for next week",

    # new title in update
    "rename my Physics exam on Monday to Physics Final",

    # duration without trigger word
    "Java OOP 3 weeks starting tomorrow",

    # show roadmap vague
    "what is my Python situation right now",
    "show me my productive hours on monday"
]

print("\nDifficult Test Cases:")
print("=" * 60)
for i, text in enumerate(test_inputs):
    result = predict_intent(text)
    status = "✓" if result["above_threshold"] else "⚠ LOW CONF"
    print(f"[{i+1}] [{status}] {result['text']}")
    print(f"       Intent     : {result['intent']}")
    print(f"       Confidence : {result['confidence']}")
    print("-" * 40)


Difficult Test Cases:
[1] [✓] Physics exam is on Monday at 5PM
       Intent     : add
       Confidence : 0.9981
----------------------------------------
[2] [✓] yo remove that physics thing i added yesterday
       Intent     : delete
       Confidence : 0.9998
----------------------------------------
[3] [✓] schedule a meeting with hamid on wednesday at 3pm
       Intent     : add
       Confidence : 0.9998
----------------------------------------
[4] [✓] move my Math exam on Friday to Monday at 6PM
       Intent     : update
       Confidence : 0.9998
----------------------------------------
[5] [✓] add a group study session with Ahmed and Sara on Sunday at 4PM
       Intent     : add
       Confidence : 0.9998
----------------------------------------
[6] [✓] I need to go through Python arrays starting next Monday in 2 weeks
       Intent     : add
       Confidence : 0.9979
----------------------------------------
[7] [✓] MongoDB queries next Friday 1 month
       Intent     : sh